# Prelude

## Imports

In [ ]:
import effector
import numpy as np
import pandas as pd

from interpret import set_visualize_provider, show
from interpret.glassbox import ExplainableBoostingRegressor
from interpret.provider import InlineProvider
from effector.calm.calm import CALMRegressor
from sklearn.model_selection import train_test_split
from utils.plot_utils import *
from experiments import set_random_seeds
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import os

## Configuration

In [ ]:
set_visualize_provider(InlineProvider())
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

## Helpers

In [ ]:
def run_all_models_kfold(X, y, max_depth=2):
    cv= KFold(n_splits=5,random_state=42,shuffle=True)
    scores = {
        "gam": [],
        "ga2m": [],
        "calm": []
    }
    for i,(train_index, test_index) in enumerate(cv.split(X, y)):
        print(f"Fold {i+1}")
        set_random_seeds(42)

        X_train_fold, X_test_fold = X[train_index], X[test_index]
        y_train_fold, y_test_fold = y[train_index], y[test_index]
        
        # GAM
        gam = ExplainableBoostingRegressor(interactions=0, random_state=42)
        gam.fit(X_train_fold, y_train_fold)
        gam_score = gam.score(X_test_fold, y_test_fold)
        scores["gam"].append(gam_score)

        # GA2M
        ga2m = ExplainableBoostingRegressor(random_state=42)
        ga2m.fit(X_train_fold, y_train_fold)
        ga2m_score = ga2m.score(X_test_fold, y_test_fold)
        scores["ga2m"].append(ga2m_score)

        # CALM
        calm = CALMRegressor()
        calm.fit(X_train_fold, y_train_fold, max_depth=max_depth)

        calm_score = calm.score(X_test_fold, y_test_fold)
        scores["calm"].append(calm_score)

    return {k: (np.mean(v), np.std(v)) for k, v in scores.items()}

In [ ]:
all_datasets_kfold_scores = {
    "dataset": ["Regional Interaction"] + ["Regional Interaction -- 4"] + ["General Interaction"],
    "gam_mean": [],
    "ga2m_mean": [],
    "calm_mean": [],
    "gam_std": [],
    "ga2m_std": [],
    "calm_std": []
}

# Regional Interaction Ground Truth

For this example, we use a ground truth response variable which has an interaction between two features, but of the specific form that our CALM models intend to capture; i.e. the contribution of one explanatory variable changes based on whether the value of another variable is above or below a certain point.

Specifically, the response variable is generated by the formula:
$$
y = x_1^2 + \log(|x_2|) + 2 
\begin{cases}
\sin(\frac{\pi}{2} x_3) & x_2 \geq 0 \\
\cos(\frac{\pi}{2} x_3) & x_2 < 0
\end{cases}
$$

The 3 variables are again sampled uniformly and independently from $[-1, 1]$, and we generate 1000 sample data points.

In [ ]:
set_random_seeds(42)
dataset = effector.datasets.IndependentUniform(dim=3, low=-1, high=1)
x = dataset.generate_data(1_000)

In [ ]:
class RegionalGenerator1(effector.models.Base):
    def __init__(self):
        super().__init__(name=self.__class__.__name__)

    def predict(self, x: np.ndarray) -> np.ndarray:
        y = x[:, 0]**2 + np.log(np.abs(x[:, 1])) + 2 * np.where(
            x[:, 1] >= 0,
            np.sin(np.pi * x[:, 2] / 2),
            np.cos(np.pi * x[:, 2] / 2),
        )
        return y

    def jacobian(self, x: np.ndarray) -> np.ndarray:
        y = np.zeros_like(x)
        y[:, 0] = 2 * x[:, 0]
        y[:, 1] = 1 / x[:, 1]
        y[:, 2] = 2 * np.where(
            x[:, 1] >= 0,
            np.cos(np.pi * x[:, 2] / 2) * np.pi / 2,
            -np.sin(np.pi * x[:, 2] / 2) * np.pi / 2,
        )
        return y
model = RegionalGenerator1()
y = model.predict(x)
features = [f"x_{i+1}" for i in range(x.shape[1])]

After generating the data, we once again perform the standard train-test split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y)

## GAM

In [ ]:
gam = ExplainableBoostingRegressor(interactions=0, random_state=42, feature_names=features)
gam.fit(X_train, y_train)
show(gam.explain_global())

## GAM2

In [ ]:
ga2m = ExplainableBoostingRegressor(random_state=42, feature_names=features)
ga2m.fit(X_train, y_train)
show(ga2m.explain_global())

## CALM

In [ ]:
calm = CALMRegressor()
calm.fit(X_train, y_train, max_depth=1, feat_labels=features)
for k, v in calm.tree.items():
    print(f"Feature: {k}")
    v.show_full_tree()
    print()

## Visualize GAM-CALM with custom visualizations

In [ ]:
plot_calm_gam_shape_function(
    gam=gam,
    calm=calm,
    ga2m=ga2m,
    feat_idx=2,
    # save_dir="regional2",
    display_title=False,
    ga2m_fixed_figsize=True,
    feat_labels=features,
    ylim_local=True,
    format_latex=True,
    fontsize=16,
)

In [ ]:
all_scores = run_all_models_kfold(x, y, max_depth=1)
for model, (mean, std) in all_scores.items():
    all_datasets_kfold_scores[f"{model}_mean"].append(mean)
    all_datasets_kfold_scores[f"{model}_std"].append(std)

# Regional Interaction Ground Truth - 4 regions

For this example, the response variable has once again an interaction between two features in the form of regions. Here, however, we take the complexity of the model one step further, by making the contribution of feature $x_3$ have 4 different branches instead of two. Which branch it follows now depends on 2 features, instead of 1.

Specifically, the response variable is generated by the formula:
$$
y = x_1^2 + \log(|x_2|) + 2 
\begin{cases}
\sin(\frac{\pi}{2} x_3) & x_1 \geq 0, x_2 \geq 0 \\
\cos(\frac{\pi}{2} x_3) & x_1 \geq 0, x_2 < 0 \\
\sin(2 \pi x_3) & x_1 < 0, x_2 \geq 0 \\
\cos(2 \pi x_3) & x_1 < 0, x_2 < 0
\end{cases}
$$

The 3 variables are again sampled uniformly and independently from $[-1, 1]$, and we generate 1000 sample data points.

In [ ]:
set_random_seeds(42)
dataset = effector.datasets.IndependentUniform(dim=3, low=-1, high=1)
x = dataset.generate_data(1_000)
features = [f"x_{i+1}" for i in range(x.shape[1])]

In [ ]:
class RegionalGenerator2(effector.models.Base):
    def __init__(self):
        super().__init__(name=self.__class__.__name__)

    def predict(self, x: np.ndarray) -> np.ndarray:
        y = x[:, 0]**2 + np.log(np.abs(x[:, 1])) + 2 * np.where(
            x[:, 0] >= 0,
            np.where(
                x[:, 1] >= 0,
                np.sin(np.pi * x[:, 2] / 2),
                np.cos(np.pi * x[:, 2] / 2),
            ),
            np.where(
                x[:, 1] >= 0,
                np.sin(2 * np.pi * x[:, 2]),
                np.cos(2 * np.pi * x[:, 2]),
            ),
        )
        return y

    def jacobian(self, x: np.ndarray) -> np.ndarray:
        y = np.zeros_like(x)
        y[:, 0] = 2 * x[:, 0]
        y[:, 1] = 1 / x[:, 1]
        y[:, 2] = 2 * np.where(
            x[:, 0] >= 0,
            np.where(
                x[:, 1] >= 0,
                np.cos(np.pi * x[:, 2] / 2) * np.pi / 2,
                -np.sin(np.pi * x[:, 2] / 2) * np.pi / 2,
            ),
            np.where(
                x[:, 1] >= 0,
                np.cos(2 * np.pi * x[:, 2]) * 2 * np.pi,
                -np.sin(2 * np.pi * x[:, 2]) * 2 * np.pi,
            ),
        )
        return y
model = RegionalGenerator2()
y = model.predict(x)

After generating the data, we once again perform the standard train-test split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y)

## GAM

In [ ]:
gam = ExplainableBoostingRegressor(interactions=0, random_state=42, feature_names=features)
gam.fit(X_train, y_train)
show(gam.explain_global())

## GAM2

In [ ]:
ga2m = ExplainableBoostingRegressor(random_state=42, feature_names=features)
ga2m.fit(X_train, y_train)
show(ga2m.explain_global())

## CALM

In [ ]:
calm = CALMRegressor()
calm.fit(X_train, y_train, feat_labels=features)
for k, v in calm.tree.items():
    print(f"Feature: {k}")
    v.show_full_tree()
    print()

## Visualize GAM-CALM with custom visualizations

In [ ]:
plot_calm_gam_shape_function(
    gam=gam,
    calm=calm,
    ga2m=ga2m,
    feat_idx=2,
    # save_dir="regional4",
    display_title=False,
    ga2m_fixed_figsize=True,
    feat_labels=features,
    ylim_local=True,
    format_latex=True,
    fontsize=16,
)

In [ ]:
all_scores = run_all_models_kfold(x, y)
for model, (mean, std) in all_scores.items():
    all_datasets_kfold_scores[f"{model}_mean"].append(mean)
    all_datasets_kfold_scores[f"{model}_std"].append(std)

# General Interaction Ground Truth

In this example, we showcase the performance and the explanations of the GAM, GA2M and CALM models on a dataset with a general-form interaction of features. Unlike the previous two examples, which we designed specifically to demonstrate the special capabilities of the CALM models, here we will study a general interaction, to get an idea of how the models behave in a more real-like scenario.

Specifically, the response variable is generated by the formula:
$$
y = x_1^2 + \log(|x_2|) \sin(\frac{\pi}{2} x_3)
$$

The 3 variables are again sampled uniformly and independently from $[-1, 1]$, and we generate 1000 sample data points.

In [ ]:
set_random_seeds(42)
dataset = effector.datasets.IndependentUniform(dim=3, low=-1, high=1)
x = dataset.generate_data(1_000)
features = [f"x_{i+1}" for i in range(x.shape[1])]

In [ ]:
class GeneralGenerator1(effector.models.Base):
    def __init__(self):
        super().__init__(name=self.__class__.__name__)

    def predict(self, x: np.ndarray) -> np.ndarray:
        y = x[:, 0]**2 + np.log(np.abs(x[:, 1])) * np.sin(np.pi * x[:, 2] / 2)
        return y
model = GeneralGenerator1()
y = model.predict(x)

After generating the data, we once again perform the standard train-test split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y)

## GAM

In [ ]:
gam = ExplainableBoostingRegressor(interactions=0, random_state=42, feature_names=features)
gam.fit(X_train, y_train)
show(gam.explain_global())

## GAM2

In [ ]:
ga2m = ExplainableBoostingRegressor(random_state=42, feature_names=features)
ga2m.fit(X_train, y_train)
show(ga2m.explain_global())

## CALM

In [ ]:
calm = CALMRegressor()
calm.fit(X_train, y_train, feat_labels=features)
for k, v in calm.tree.items():
    print(f"Feature: {k}")
    v.show_full_tree()
    print()

## Visualize GAM-CALM with custom visualizations

In [ ]:
plot_calm_gam_shape_function(
    gam,
    calm,
    ga2m,
    1,
    # save_dir="general",
    figsize=(10, 6),
    display_title=False,
    ga2m_fixed_figsize=True,
    feat_labels=features,
    format_latex=True,
    fontsize=16
)

In [ ]:
all_scores = run_all_models_kfold(x, y)
for model, (mean, std) in all_scores.items():
    all_datasets_kfold_scores[f"{model}_mean"].append(mean)
    all_datasets_kfold_scores[f"{model}_std"].append(std)

# Final Scores

In [ ]:
scores_df = pd.DataFrame(all_datasets_kfold_scores)

for method in ["gam", "ga2m", "calm"]:
    scores_df[method] = scores_df.apply(lambda x: f"{x[method + '_mean']:.3f} ± {x[method + '_std']:.3f}", axis=1)
scores_df = scores_df.drop(columns=["gam_mean", "ga2m_mean", "calm_mean", "gam_std", "ga2m_std", "calm_std"])
scores_df = scores_df.rename(columns={
    "dataset": "Dataset",
    "gam": "GAM",
    "ga2m": "GA2M",
    "calm": "CALM"
})
display(scores_df)